In [ ]:
!pip install torch_geometric

In [ ]:
!pip install scikit-learn

In [22]:
import os
import re
import time
import json
import math
import torch
import requests
import numpy as np
import pandas as pd
import seaborn as sns
import torch.nn as nn
from enum import Enum
from tqdm import tqdm
from typing import Dict
import torch.optim as optim
from collections import Counter
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch_geometric.transforms as T
from torch_geometric.data import HeteroData
from torch.utils.data import Dataset, DataLoader
from torch_geometric.nn import SAGEConv, to_hetero
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


TEXT to VECTOR

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
import gc
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

# 1. CẤU HÌNH ĐƯỜNG DẪN MỚI (DÀNH CHO SYNTHETIC DATA)
CV_COMBINED_SYNTH_PATH = '/kaggle/input/datasets/kevinhieutmtpv/score-v1/CV_Dataset_FAB.csv'

# Đường dẫn đến JD Master (Chỉ cần JD, vì CV đã nằm sẵn trong file Synthetic)
JD_MASTER_PATH = '/kaggle/input/datasets/kevinhieutmtpv/score-v1/JD_Dataset_FAB.csv'

MODELS_CONFIG = {
    "MPNet": "all-mpnet-base-v2",
    "JobBERT": "jjzha/jobbert-base-cased",
    "BGE": "BAAI/bge-base-en-v1.5",
    "E5": "intfloat/e5-base-v2",
    "MultiQA": "sentence-transformers/multi-qa-mpnet-base-dot-v1"
}

# =====================================================================
# 2. HÀM ÉP KIỂU VECTOR CHO SYNTHETIC DATA
# =====================================================================
def generate_embeddings(domain_name, synthetic_label_path, jd_path, embedding_model, model_alias):
    print(f"\n[{model_alias}] -> Bắt đầu xử lý Text cho: {domain_name}")
    
    out_jd_npy = f'/kaggle/working/JD_Embeddings_{domain_name}_{model_alias}.npy'
    out_cv_npy = f'/kaggle/working/CV_Embeddings_{domain_name}_{model_alias}.npy'
    out_edge_csv = f'/kaggle/working/Edge_Index_{domain_name}_{model_alias}.csv'
    
    # 1. Đọc Synthetic Data (Đã có sẵn TITLE_CV, DESCRIPTION_CV, EXPERIENCE_CV)
    df_final = pd.read_csv(synthetic_label_path, dtype=str)
    
    # 2. Đọc JD Data và đổi tên cột để tránh trùng lặp
    cols_to_use = ['TITLE', 'DESCRIPTION', 'EXPERIENCE', 'SKILLS']
    df_jd = pd.read_csv(jd_path)[['Job_ID'] + cols_to_use].fillna("")
    df_jd = df_jd.rename(columns={
        'TITLE': 'TITLE_JD', 
        'DESCRIPTION': 'DESCRIPTION_JD', 
        'EXPERIENCE': 'EXPERIENCE_JD', 
        'SKILLS': 'SKILLS_JD'
    })

    # Hàm bọc thép ID
    def clean_id_advanced(col):
        def fix_format(val):
            val_str = str(val).strip().upper()
            try:
                if val_str.replace('.', '', 1).replace('E+', '', 1).isdigit():
                    return str(int(float(val_str)))
            except:
                pass
            return val_str 
        return col.apply(fix_format)

    df_final['Job_ID'] = clean_id_advanced(df_final['Job_ID'])
    df_jd['Job_ID'] = clean_id_advanced(df_jd['Job_ID'])
    
    df_jd = df_jd.drop_duplicates(subset=['Job_ID'], keep='first')
    
    # 3. CHỈ MERGE JD (Không Merge CV vì CV đã nằm sờ sờ trong df_final)
    df_merged = pd.merge(df_final, df_jd, on='Job_ID', how='left')
    
    # Ép kiểu an toàn chống lỗi 'float' object is not subscriptable
    df_merged = df_merged.fillna("").astype(str)
    
    # 4. TÁI TẠO TEXT (Nhồi mỏ vàng DESCRIPTION_CV vào)
    df_merged['JD_Text'] = df_merged['TITLE_JD'] + "\n" + df_merged['DESCRIPTION_JD'] + "\n" + df_merged['EXPERIENCE_JD'] + "\n" + df_merged['SKILLS_JD']
    df_merged['CV_Text'] = df_merged['TITLE_CV'] + "\n" + df_merged['DESCRIPTION_CV'] + "\n" + df_merged['EXPERIENCE_CV']
    
    # Cắt gọn Text
    jd_texts = df_merged['JD_Text'].str.slice(0, 1500).tolist()
    cv_texts = df_merged['CV_Text'].str.slice(0, 2500).tolist()
    
    # 5. Nhúng văn bản
    jd_embeddings = embedding_model.encode(jd_texts, batch_size=64, show_progress_bar=True)
    cv_embeddings = embedding_model.encode(cv_texts, batch_size=64, show_progress_bar=True)
    
    # Lưu trữ
    np.save(out_jd_npy, jd_embeddings)
    np.save(out_cv_npy, cv_embeddings)
    df_merged[['Job_ID', 'Resume_ID', 'LLM_Score']].to_csv(out_edge_csv, index=False)
    
    print(f"[{model_alias}] -> Hoàn tất {domain_name}! Kích thước: {cv_embeddings.shape}")

# =====================================================================
# 3. THỰC THI LỆNH
# =====================================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\n🚀 CHẠY TRÊN THIẾT BỊ: {device.upper()}")

for model_alias, model_repo in MODELS_CONFIG.items():
    print(f"\nĐANG TẢI MÔ HÌNH: {model_alias} ({model_repo})")
    
    current_model = SentenceTransformer(model_repo, device=device)
    
    # Chỉ cần chạy 1 lần cho tập Synthetic IT tổng
    generate_embeddings("SYNTH_IT", CV_COMBINED_SYNTH_PATH, JD_MASTER_PATH, current_model, model_alias)
    
    # Giải phóng VRAM
    del current_model
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()
        
print("\n✅ ĐÃ HOÀN TẤT XUẤT VECTOR CHO CẢ 5 MÔ HÌNH TỪ TẬP DATA GIẢ LẬP!")

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import HeteroData
import os

# 1. HÀM XÂY DỰNG ĐỒ THỊ HAI PHÍA (BIPARTITE GRAPH BUILDER)
def build_bipartite_graph(domain_name, model_alias, base_dir):
    print(f"KHỞI TẠO ĐỒ THỊ KHÔNG GIAN CHO NGÀNH: {domain_name}| LÕI NHÚNG: {model_alias}")
    
    # --- ĐỊNH TUYẾN ĐƯỜNG DẪN ---
    edge_csv = f'{base_dir}/Edge_Index_SYNTH_{domain_name}_{model_alias}.csv'
    jd_npy = f'{base_dir}/JD_Embeddings_SYNTH_{domain_name}_{model_alias}.npy'
    cv_npy = f'{base_dir}/CV_Embeddings_SYNTH_{domain_name}_{model_alias}.npy'
    
    # Kiểm tra tính toàn vẹn của file
    for path in [edge_csv, jd_npy, cv_npy]:
        if not os.path.exists(path):
            print(f"❌ Lỗi hệ thống: Không tìm thấy {path}")
            return None

    # --- BƯỚC 1: TẢI DỮ LIỆU THÔ ---
    print("-> Đang nạp ma trận vector và danh sách liên kết...")
    edges_df = pd.read_csv(edge_csv, dtype=str)
    jd_embs_raw = np.load(jd_npy)
    cv_embs_raw = np.load(cv_npy)
    
    # Ép kiểu điểm số (Soft Label) về Float Tensor chuẩn cho hàm Loss
    edges_df['LLM_Score'] = pd.to_numeric(edges_df['LLM_Score'], errors='coerce')
    
    # --- BƯỚC 2: KHỬ TRÙNG LẶP VÀ XÂY DỰNG KHÔNG GIAN ĐỈNH (NODES) ---
    print("-> Đang quy hoạch hệ tọa độ Đỉnh (Node Mapping)...")
    
    # Trích xuất các ID độc nhất và vị trí xuất hiện đầu tiên của chúng
    unique_jd_ids, jd_indices = np.unique(edges_df['Job_ID'], return_index=True)
    unique_cv_ids, cv_indices = np.unique(edges_df['Resume_ID'], return_index=True)
    
    # Lấy vector đại diện duy nhất cho mỗi Đỉnh
    jd_node_features = jd_embs_raw[jd_indices]
    cv_node_features = cv_embs_raw[cv_indices]
    
    # Tạo từ điển ánh xạ (Global ID -> Tensor Index [0, N-1])
    jd_mapping = {jid: i for i, jid in enumerate(unique_jd_ids)}
    cv_mapping = {cid: i for i, cid in enumerate(unique_cv_ids)}
    
    # --- BƯỚC 3: DỆT CẠNH (EDGES) VÀ TRỌNG SỐ ---
    print("-> Đang đúc kết cấu trúc cạnh (Edge Index)...")
    src_nodes = [jd_mapping[jid] for jid in edges_df['Job_ID']]
    dst_nodes = [cv_mapping[cid] for cid in edges_df['Resume_ID']]
    
    # Cấu trúc tensor cho PyG bắt buộc phải có shape [2, num_edges]
    edge_index = torch.tensor([src_nodes, dst_nodes], dtype=torch.long)
    edge_labels = torch.tensor(edges_df['LLM_Score'].values, dtype=torch.float32)
    
    # --- BƯỚC 4: LẮP RÁP HETERODATA (ĐỒ THỊ DỊ THỂ) ---
    print("-> Đang đóng gói vào cấu trúc PyTorch Geometric HeteroData...")
    data = HeteroData()
    
    # Khai báo Nút
    data['job'].x = torch.tensor(jd_node_features, dtype=torch.float32)
    data['resume'].x = torch.tensor(cv_node_features, dtype=torch.float32)
    
    # Khai báo Cạnh (Job kết nối với Resume thông qua quan hệ 'requires')
    data['job', 'requires', 'resume'].edge_index = edge_index
    data['job', 'requires', 'resume'].edge_label = edge_labels
    
    print("="*70)
    print(f"✅ ĐÃ DỆT THÀNH CÔNG ĐỒ THỊ [{domain_name} - {model_alias}]!")
    print("="*70)
    
    # Lưu Graph Object ra file vật lý để xài cho các buổi train sau
    output_graph_path = f'/kaggle/working/Graph_Data_{domain_name}_{model_alias}.pt'
    torch.save(data, output_graph_path)
    print(f"💾 Đồ thị đã được bảo lưu tại: {output_graph_path}")
    
    return data

# THỰC THI
BASE_EMBEDDING_DIR = '/kaggle/input/datasets/kevinhieutmtpv/embedings'

DOMAINS = ["IT"]
MODELS = ["MPNet", "JobBERT", "BGE", "E5", "MultiQA"]

# Dictionary chứa toàn bộ 10 đồ thị để dễ dàng gọi ra huấn luyện sau này
all_graphs = {}

print(f"🚀 KHỞI ĐỘNG DÂY CHUYỀN LẮP RÁP 10 ĐỒ THỊ TỪ THƯ MỤC: {BASE_EMBEDDING_DIR}")

for domain in DOMAINS:
    all_graphs[domain] = {}
    for model in MODELS:
        graph = build_bipartite_graph(domain, model, BASE_EMBEDDING_DIR)
        if graph is not None:
            all_graphs[domain][model] = graph

print("\n🏆 QUÁ TRÌNH DỆT ĐỒ THỊ HOÀN TẤT. SẴN SÀNG ĐƯA VÀO GNN!")

In [ ]:
import torch
import torch_geometric.transforms as T
import os

# =====================================================================
# HÀM CẮT LỚP ĐỒ THỊ BẢO TOÀN CẤU TRÚC (LINK SPLITTING)
# =====================================================================
def split_graph_for_training(domain_name, model_alias, base_dir):
    print(f"🔪 KHỞI CHẠY CHIẾN LƯỢC PHÂN RÃ ĐỒ THỊ | NGÀNH: {domain_name} - LÕI: {model_alias}")
    
    # 1. Tải Đồ thị gốc lên
    graph_path = f'{base_dir}/Graph_Data_{domain_name}_{model_alias}.pt'
    if not os.path.exists(graph_path):
        print(f"❌ Không tìm thấy đồ thị tại {graph_path}")
        return
        
    data = torch.load(graph_path, weights_only=False)
    print("-> Đã nạp Đồ thị gốc thành công.")
    
    # 2. Khởi tạo lưỡi dao RandomLinkSplit của PyG
    # - num_val: 10% số cạnh dành cho Validation (Xác thực lúc train)
    # - num_test: 10% số cạnh dành cho Test (Kiểm tra độc lập cuối cùng)
    # - is_undirected: False (Đồ thị của ta có hướng từ Job -> Resume)
    # - edge_types: Chỉ định đích danh loại cạnh cần cắt
    # - rev_edge_types: Không dùng cạnh ngược để tránh rò rỉ dữ liệu (Data Leakage)
    
    transform = T.RandomLinkSplit(
        num_val=0.1,
        num_test=0.1,
        is_undirected=False,
        edge_types=[('job', 'requires', 'resume')],
        rev_edge_types=None,
        add_negative_train_samples=False, # Chúng ta đã tính toán sẵn Label 0 rồi, không cần PyG sinh thêm rác!
        neg_sampling_ratio=0.0
    )
    
    # 3. Tiến hành chém đồ thị
    print("-> Đang thực thi thuật toán chia tách (80% Train - 10% Val - 10% Test)...")
    train_data, val_data, test_data = transform(data)
    
    # 4. Lưu lại 3 "mảnh vỡ" này vào ổ cứng
    out_dir = '/kaggle/working/splits'
    os.makedirs(out_dir, exist_ok=True)
    
    torch.save(train_data, f'{out_dir}/Train_{domain_name}_{model_alias}.pt')
    torch.save(val_data, f'{out_dir}/Val_{domain_name}_{model_alias}.pt')
    torch.save(test_data, f'{out_dir}/Test_{domain_name}_{model_alias}.pt')
    
    print(f"✅ PHÂN RÃ THÀNH CÔNG [{domain_name} - {model_alias}]!")

# Thực thi cho cả 2 ngành
GRAPH_BASE_DIR = '/kaggle/working/'
DOMAINS = ["IT"]
MODELS = ["MPNet", "JobBERT", "BGE", "E5", "MultiQA"]

print(f"KHỞI ĐỘNG DÂY CHUYỀN PHÂN RÃ 10 ĐỒ THỊ TỪ: {GRAPH_BASE_DIR}")

for domain in DOMAINS:
    for model in MODELS:
        split_graph_for_training(domain, model, GRAPH_BASE_DIR)

print("\nQUÁ TRÌNH PHÂN RÃ HOÀN TẤT. SẴN SÀNG CHO BƯỚC HUẤN LUYỆN GNN!")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, to_hetero, GATConv

class LocalMatch(nn.Module):
    """
    So sánh độc lập 2 vector (VD: Skill của Job vs Skill của CV).
    Bắt chước chính xác class LocalMatch của MUFFIN.
    """
    def __init__(self, hidden_channels):
        super(LocalMatch, self).__init__()
        # Ghép 4 thành phần: [A, B, A-B, A*B] -> Kích thước x4
        insize = 4 * hidden_channels
        outsize = hidden_channels
        
        self.net = nn.Sequential(
            nn.Linear(insize, insize),
            nn.PReLU(),
            nn.Linear(insize, outsize),
            nn.PReLU()
        )

    def forward(self, a, b):
        # Tính khoảng cách và góc tương đồng
        diff = torch.abs(a - b)
        prod = a * b
        
        # Nối lại theo công thức của MUFFIN
        c = torch.cat([a, b, diff, prod], dim=-1)
        return self.net(c)

# =====================================================================
# 1. BỘ MÃ HÓA ĐỒ THỊ (GNN ENCODER)
# =====================================================================
class GNNEncoder(nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), out_channels)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

class EdgePredictor_V1_5(nn.Module):
    def __init__(self, hidden_channels):
        super(EdgePredictor_V1_5, self).__init__()
        # Muffin chuẩn: Ghép 4 thành phần [Job, CV, Job - CV, Job * CV] 
        # nên đầu vào là 4 * hidden_channels
        insize = 4 * hidden_channels
        outsize = hidden_channels
        
        # Cấu trúc mạng học tương tác bắt chước LocalMatch [1]
        self.net = nn.Sequential(
            nn.Linear(insize, insize),
            nn.PReLU(),
            nn.Linear(insize, outsize),
            nn.PReLU(),
            # Thêm lớp chiếu cuối cùng để quy về 1 điểm số (Ranking Score)
            nn.Linear(outsize, 1),
            # Do điểm ranking của LLM (Gemini/GPT) chấm nằm trong dải [5]
            # nên thêm Sigmoid để chặn khoảng giá trị đầu ra
            nn.Sigmoid() 
        )

    def forward(self, job_emb, cv_emb):
        # 1. Tính khoảng cách (Hiệu) và Độ tương đồng góc (Tích) theo chuẩn MUFFIN [1]
        diff = job_emb - cv_emb
        prod = job_emb * cv_emb
        
        # 2. Nối 4 ma trận đặc trưng
        c = torch.cat([job_emb, cv_emb, diff, prod], dim=-1)
        
        # 3. Đưa qua mạng PReLU để dự đoán
        score = self.net(c)
        
        return score.squeeze(-1)

# 2. BỘ GIẢI MÃ VÀ DỰ ĐOÁN ĐIỂM (EDGE PREDICTOR)
class EdgePredictor_Regression_V1_5(nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        input_dim = 4 * hidden_channels + 1
        
        self.lin1 = nn.Linear(input_dim, hidden_channels)
        self.lin2 = nn.Linear(hidden_channels, 1)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x_i, x_j):
        # 1. Tính Tích từng phần tử (Element-wise product) - Bắt góc tương giao
        prod = x_i * x_j
        
        # 2. Tính Hiệu khoảng cách tuyệt đối (Element-wise absolute difference)
        diff = torch.abs(x_i - x_j)
        
        # 3. Tính độ tương đồng Cosine (Từ -1.0 đến 1.0)
        # Ép thêm chiều unsqueeze(-1) để nó từ dạng mảng 1D thành cột 2D ghép vào cho khớp
        cos_sim = F.cosine_similarity(x_i, x_j, dim=-1).unsqueeze(-1)
        
        # 4. Gộp tất cả các luồng thông tin "mớm tận miệng" này lại
        x = torch.cat([x_i, x_j, prod, diff, cos_sim], dim=-1)
        
        # 5. Đưa qua mạng nơ-ron phân loại
        x = self.lin1(x).relu()
        x = self.dropout(x)
        return torch.sigmoid(self.lin2(x))
# 3. LẮP RÁP HỆ THỐNG PJFCANN TỔNG THỂ
class PJFCANN_Regression(nn.Module):
    def __init__(self, hidden_channels, metadata):
        super().__init__()
        # 1. Bộ mã hóa cấu trúc đồ thị
        self.encoder = to_hetero(GNNEncoder(hidden_channels, hidden_channels), metadata=metadata, aggr='sum')
        
        self.decoder = EdgePredictor_Regression_V1_5(hidden_channels)

    def forward(self, x_dict, edge_index_dict, edge_label_index):
        # 1. Đưa toàn bộ cấu trúc đồ thị vào bộ mã hóa
        z_dict = self.encoder(x_dict, edge_index_dict)
        
        # 2. Bóc tách vector của Đỉnh Job và Đỉnh CV dựa trên các cạnh cần dự đoán
        row, col = edge_label_index
        z_job = z_dict['job'][row]
        z_resume = z_dict['resume'][col]
        
        score = self.decoder(z_job, z_resume).squeeze(-1)
        return score

# 3. MÔ HÌNH CHÍNH TÍCH HỢP: PJFCANN + MUFFIN (Giải quyết Lỗi 2 & 3)
# =====================================================================
class PJFCANN_MUFFIN_Hybrid(nn.Module):
    def __init__(self, hidden_channels, metadata):
        super().__init__()
        
        # 1. GNN để tạo H_global
        self.encoder = to_hetero(GNNEncoder(hidden_channels, hidden_channels), metadata=metadata, aggr='sum')

        # 2. Các bộ LocalMatch độc lập cho từng trường (H_local) - Chuẩn MUFFIN
        self.match_skills = LocalMatch(hidden_channels)
        self.match_experience = LocalMatch(hidden_channels)
        self.match_description = LocalMatch(hidden_channels)
        
        # 3. Bộ LocalMatch cho thông tin Đồ thị (H_global)
        self.match_global = LocalMatch(hidden_channels)

        # 4. Lớp chấm điểm cuối cùng (MLP)
        # Tổng hợp từ 4 nguồn: 3 trường ngữ nghĩa cục bộ + 1 biểu diễn đồ thị toàn cục
        self.scoring_mlp = nn.Sequential(
            nn.Linear(4 * hidden_channels, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1), # Đầu ra 1 chiều (Điểm Ranking)
            nn.Sigmoid()
        )

    def forward(self, x_dict, edge_index_dict, job_idx, cv_idx, 
                job_skills, cv_skills, 
                job_exp, cv_exp, 
                job_desc, cv_desc):
        """
        job_idx, cv_idx: mảng chỉ mục các cạnh cần dự đoán.
        job_skills, cv_skills, ... : Tensor embedding riêng biệt cho từng trường (đã được nhúng bằng MPNet/BERT).
        """
        
        # --- BƯỚC 1: LẤY BIỂU DIỄN TOÀN CỤC (H_global) TỪ GNN ---
        # x_dict có thể là trung bình cộng của các trường (skills, exp, desc) để làm Node Feature khởi tạo cho đồ thị.
        h_global_dict = self.encoder(x_dict, edge_index_dict)
        h_global_job = h_global_dict['job'][job_idx]
        h_global_cv = h_global_dict['resume'][cv_idx]

        # --- BƯỚC 2: MULTI-FIELD LOCAL MATCHING (Sửa Lỗi 2: Tránh gom rác) ---
        # So sánh chéo từng trường độc lập từ text nhúng ban đầu (H_local)
        matched_skills = self.match_skills(job_skills[job_idx], cv_skills[cv_idx])
        matched_exp = self.match_experience(job_exp[job_idx], cv_exp[cv_idx])
        matched_desc = self.match_description(job_desc[job_idx], cv_desc[cv_idx])

        # --- BƯỚC 3: TƯƠNG TÁC ĐỒ THỊ TOÀN CỤC ---
        matched_global = self.match_global(h_global_job, h_global_cv)

        # --- BƯỚC 4: GHÉP NỐI CỤC BỘ & TOÀN CỤC (Sửa Lỗi 3: H_J = [H_local; H_global]) ---
        # Thay vì chỉ dùng H_global, ta nối kết quả của cả 3 H_local và 1 H_global
        combined_features = torch.cat([matched_skills, matched_exp, matched_desc, matched_global], dim=-1)

        # Đưa qua mạng nơ-ron để ra điểm xếp hạng (Ranking Score)
        score = self.scoring_mlp(combined_features)
        
        return score.squeeze(-1)
print("✅ Đã nạp kiến trúc PJFCANN_Regression vào bộ nhớ ảo!")

In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import math
import torch_geometric.transforms as T
import os
import matplotlib.pyplot as plt
import seaborn as sns
from torch.optim.lr_scheduler import ReduceLROnPlateau
# 1. CẤU HÌNH HỆ THỐNG ĐÃ NÂNG CẤP
HIDDEN_CHANNELS = 128
LEARNING_RATE = 0.005
WEIGHT_DECAY = 1e-4

EPOCHS = 500 
PATIENCE = 40 
LR_PATIENCE = 10 
LR_FACTOR = 0.5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

undirected_transform = T.ToUndirected()
criterion = torch.nn.MSELoss()

# ĐỊNH TUYẾN ĐƯỜNG DẪN
SPLITS_DIR = '/kaggle/working/splits'
GLOBAL_TRAINED_MODELS = {}
# =====================================================================
# 2. HÀM HUẤN LUYỆN (TÍCH HỢP EARLY STOPPING)
# =====================================================================
def train_gnn(domain_name, model_alias):
    print(f"KHỞI ĐỘNG LÒ PHẢN ỨNG | NGÀNH: {domain_name} - LÕI: {model_alias}")

    OUTPUT_DIR = '/kaggle/working/PJFCANN_Data/'
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
        
    MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, f'pjfcann_regression_{domain_name}_{model_alias}_best.pth')
    
    train_path = f'{SPLITS_DIR}/Train_{domain_name}_{model_alias}.pt'
    val_path = f'{SPLITS_DIR}/Val_{domain_name}_{model_alias}.pt'
    
    if not os.path.exists(train_path) or not os.path.exists(val_path):
        print(f"Lỗi: Không tìm thấy dữ liệu split cho {domain_name} - {model_alias}")
        return

    train_data = torch.load(train_path, weights_only=False).to(device)
    val_data = torch.load(val_path, weights_only=False).to(device)
    
    train_data = undirected_transform(train_data)
    val_data = undirected_transform(val_data)

    model = PJFCANN_Regression(hidden_channels=HIDDEN_CHANNELS, metadata=train_data.metadata()).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE)
    
    best_val_rmse = float('inf')
    best_model_path = f'/kaggle/working/Best_PJFCANN_{domain_name}_{model_alias}.pth'
    
    epochs_no_improve = 0 # Bộ đếm số lần không tiến bộ

    for epoch in range(1, EPOCHS + 1):
        current_lr = optimizer.param_groups[0]['lr']
        # --- BƯỚC 1: TRAIN ---
        model.train()
        optimizer.zero_grad()
        
        edge_label_index = train_data['job', 'requires', 'resume'].edge_label_index
        edge_label = train_data['job', 'requires', 'resume'].edge_label
        
        out = model(train_data.x_dict, train_data.edge_index_dict, edge_label_index)
        
        loss = criterion(out, edge_label.float())
        loss.backward()
        optimizer.step()
        
        # --- BƯỚC 2: VALIDATION (CHẠY MỖI EPOCH ĐỂ THEO DÕI SÁT SAO) ---
        model.eval()
        with torch.no_grad():
            val_idx = val_data['job', 'requires', 'resume'].edge_label_index
            val_lbl = val_data['job', 'requires', 'resume'].edge_label
            val_out = model(val_data.x_dict, val_data.edge_index_dict, val_idx)
            
            preds = val_out.cpu().numpy()
            targets = val_lbl.cpu().numpy()
            
            val_rmse = math.sqrt(mean_squared_error(targets, preds))
            val_mae = mean_absolute_error(targets, preds)

        scheduler.step(val_rmse)
        
        # --- BƯỚC 3: KIỂM TRA ĐIỀU KIỆN KÍCH HOẠT CẦU DAO ---
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            torch.save(model.state_dict(), best_model_path)
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            epochs_no_improve = 0 # Reset bộ đếm nếu có kỷ lục mới
        else:
            epochs_no_improve += 1
            
        if epochs_no_improve >= PATIENCE:
            print(f"🛑 [EARLY STOPPING] Kích hoạt cầu dao tự động tại Epoch {epoch}!")
            print(f"-> Lý do: Val RMSE không giảm trong {PATIENCE} epochs liên tiếp.")
            break # Ngắt ngay lò phản ứng
        global GLOBAL_TRAINED_MODELS
        GLOBAL_TRAINED_MODELS[f"{domain_name}_{model_alias}"] = model
    print(f"🎉 HOÀN TẤT TRAIN [{domain_name} - {model_alias}]! Best Val RMSE: {best_val_rmse:.4f}")

# =====================================================================
# 3. HÀM KIỂM ĐỊNH CHUNG KẾT VÀ VẼ BIỂU ĐỒ (GIỮ NGUYÊN)
# =====================================================================
def evaluate_and_plot(domain_name, model_alias):
    print(f"🏆 CHUNG KẾT: KIỂM ĐỊNH TRÊN TẬP TEST | {domain_name} - {model_alias}")
    
    test_path = f'{SPLITS_DIR}/Test_{domain_name}_{model_alias}.pt'
    if not os.path.exists(test_path):
        print(f"Lỗi: Không tìm thấy file Test tại {test_path}")
        return
        
    test_data = torch.load(test_path, weights_only=False).to(device)
    test_data = undirected_transform(test_data)
    
    model = PJFCANN_Regression(hidden_channels=HIDDEN_CHANNELS, metadata=test_data.metadata()).to(device)
    
    model_path = f'/kaggle/working/Best_PJFCANN_{domain_name}_{model_alias}.pth'
    if not os.path.exists(model_path):
        print(f"Không tìm thấy mô hình vàng tại {model_path}")
        return
        
    model.load_state_dict(torch.load(model_path))
    model.eval()
    
    with torch.no_grad():
        edge_label_index = test_data['job', 'requires', 'resume'].edge_label_index
        edge_label = test_data['job', 'requires', 'resume'].edge_label
        out = model(test_data.x_dict, test_data.edge_index_dict, edge_label_index)
        
        preds = out.cpu().numpy()
        targets = edge_label.cpu().numpy()
        
        test_rmse = math.sqrt(mean_squared_error(targets, preds))
        test_mae = mean_absolute_error(targets, preds)
        test_r2 = r2_score(targets, preds)
        
    print(f"📊 KẾT QUẢ TẬP TEST:")
    print(f"-> Test RMSE: {test_rmse:.4f}")
    print(f"-> Test MAE : {test_mae:.4f}")
    print(f"-> Test R2  : {test_r2:.4f}")
    
    # print(f"🎨 Đang xuất bản đồ họa Scatter Plot...")
    # sns.set_theme(style="whitegrid")
    # plt.figure(figsize=(8, 8))

    # plt.scatter(targets, preds, alpha=0.6, color='teal', edgecolor='k', s=45, label='Predicted Points')
    # plt.plot([-0.05, 1.05], [-0.05, 1.05], color='crimson', linestyle='--', linewidth=2.5, label='Perfect Prediction (y=x)')

    # plt.title(f'GNN Regression ({domain_name} - {model_alias})\nActual vs Predicted Scores', fontsize=14, fontweight='bold', pad=20)
    # plt.xlabel('Actual Expert Score (0.0 to 1.0)', fontsize=12, fontweight='bold')
    # plt.ylabel('Predicted Model Score (0.0 to 1.0)', fontsize=12, fontweight='bold')

    # plt.xlim(-0.05, 1.05)
    # plt.ylim(-0.05, 1.05)

    # bbox_props = dict(boxstyle="round,pad=0.5", fc="white", ec="gray", lw=1.5, alpha=0.9)
    # plt.text(0.05, 0.95, f"Test Metrics\n-------------------\nMAE:   {test_mae:.4f}\nRMSE: {test_rmse:.4f}\nR²:      {test_r2:.4f}",
    #          fontsize=11, fontweight='bold', transform=plt.gca().transAxes,
    #          verticalalignment='top', bbox=bbox_props, family='monospace')

    # plt.legend(loc='lower right', fontsize=11)

    # output_img_path = f'/kaggle/working/Figure5_Actual_vs_Predicted_{domain_name}_{model_alias}.png'
    # plt.savefig(output_img_path, dpi=300, bbox_inches='tight')
    # print(f"✅ Đã lưu biểu đồ thành công: {output_img_path}")
    # plt.show()

# =====================================================================
# 4. THỰC THI TOÀN BỘ PIPELINE
# =====================================================================
DOMAINS = ["IT", "HR"]
MODELS = ["MPNet", "JobBERT", "BGE", "E5", "MultiQA"]

print(f"🚀 BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN VÀ KIỂM ĐỊNH (EPOCHS: {EPOCHS} | PATIENCE: {PATIENCE})")

for domain in DOMAINS:
    for model_alias in MODELS:
        train_gnn(domain, model_alias)
        evaluate_and_plot(domain, model_alias)

print("\n🎉 ĐÃ HOÀN TẤT TOÀN BỘ PIPELINE THỰC NGHIỆM MỚI!")

In [ ]:
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
import math
import os
import torch_geometric.transforms as T
print("🔥 KHỞI ĐỘNG ĐẤU TRƯỜNG AI: SO SÁNH GNN VỚI 4 MÔ HÌNH SOTA BASELINE")
# 1. HÀM TRÍCH XUẤT DỮ LIỆU ĐỂ ĐẢM BẢO "CÔNG BẰNG TUYỆT ĐỐI"
def extract_features_for_baselines(graph_data):
    """
    Hàm này bóc tách ma trận vector thô từ PyG Graph.
    Nó tái tạo chính xác công thức V1.5: [Job, CV, Tích, Hiệu tuyệt đối, Cosine]
    nhưng KHÔNG qua lớp chập SAGEConv (để xem GNN lợi hại cỡ nào).
    """
    job_idx = graph_data['job', 'requires', 'resume'].edge_label_index[0].numpy()
    cv_idx = graph_data['job', 'requires', 'resume'].edge_label_index[1].numpy()

    x_job = graph_data['job'].x[job_idx].numpy()
    x_cv = graph_data['resume'].x[cv_idx].numpy()
    y = graph_data['job', 'requires', 'resume'].edge_label.numpy()

    # Tính toán các đặc trưng kỹ nghệ (Feature Engineering)
    prod = x_job * x_cv
    diff = np.abs(x_job - x_cv)

    # Tính Cosine Similarity (Vectorized)
    num = np.sum(x_job * x_cv, axis=1)
    den = np.linalg.norm(x_job, axis=1) * np.linalg.norm(x_cv, axis=1)
    cos_sim = (num / den).reshape(-1, 1)

    # Ghép nối thành ma trận 2D cực lớn
    X = np.concatenate([x_job, x_cv, prod, diff, cos_sim], axis=1)
    return X, y
    
# 2. KHỞI TẠO 4 MÔ HÌNH ĐỐI CHỨNG
# Khai báo các mô hình tiêu chuẩn hiện đại
models = {
    "1. Random Forest (Ensemble)": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "2. XGBoost Regressor (SOTA)": xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, random_state=42, tree_method='hist'),
    "3. LightGBM Regressor (Microsoft)": lgb.LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "4. MLP Regressor (Deep Learning)": MLPRegressor(hidden_layer_sizes=(128, 64), activation='relu', max_iter=200, random_state=42)
}

# 3. TIẾN HÀNH KIỂM ĐỊNH (NGÀNH HR HOẶC IT)
def run_arena(domain_name, model_alias):
    print(f"🏆 CHẠY SO SÁNH ĐỐI CHỨNG CHO NGÀNH: {domain_name}")
    # Load dữ liệu đã chia (Dùng chung đúng file Test của GNN để so sánh)
    SPLITS_DIR = '/kaggle/working/splits' # Cậu nhớ sửa link này cho khớp môi trường

    train_path = f'{SPLITS_DIR}/Train_{domain_name}_{model_alias}.pt'
    test_path = f'{SPLITS_DIR}/Test_{domain_name}_{model_alias}.pt'

    if not os.path.exists(train_path) or not os.path.exists(test_path):
        print(f"❌ Lỗi: Không tìm thấy dữ liệu split cho {domain_name} - {model_alias}")
        return

    train_data = torch.load(train_path, weights_only=False)
    test_data = torch.load(test_path, weights_only=False)

    X_train, y_train = extract_features_for_baselines(train_data)
    X_test, y_test = extract_features_for_baselines(test_data)

    results = []
    # Huấn luyện và dự đoán từng mô hình
    for name, model in models.items():
        print(f"⏳ Đang huấn luyện {name}...")

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        # Cắt gọt điểm số để không bị lố ra khỏi [0, 1]
        preds = np.clip(preds, 0.0, 1.0)

        rmse = math.sqrt(mean_squared_error(y_test, preds))
        mae = mean_absolute_error(y_test, preds)
        r2 = r2_score(y_test, preds)

        results.append({"Model": name, "RMSE": round(rmse, 4), "MAE": round(mae, 4), "R2-Score": round(r2, 4)})

    print(f"🧠 Đang triệu hồi GNN Đề xuất (PJFCANN) để chấm điểm đối chứng...")
    test_data_gnn = undirected_transform(test_data.to(device))

    gnn_model = PJFCANN_Regression(hidden_channels=128, metadata=test_data_gnn.metadata()).to(device)
    # Load trọng số vừa mới train lúc nãy (Sử dụng model đã ép xung nếu có, không thì load model thường)
    best_gnn_path = f'/kaggle/working/Best_PJFCANN_{domain_name}_{model_alias}.pth'

    if not os.path.exists(best_gnn_path):
        print(f"⚠️ Không tìm thấy {best_gnn_path}. Hãy chắc chắn cậu đã train GNN trước khi chạy so sánh!")
    else:
        gnn_model.load_state_dict(torch.load(best_gnn_path))
        gnn_model.eval()

        with torch.no_grad():

            edge_idx = test_data_gnn['job', 'requires', 'resume'].edge_label_index
            y_true_gnn = test_data_gnn['job', 'requires', 'resume'].edge_label

            gnn_out = gnn_model(test_data_gnn.x_dict, test_data_gnn.edge_index_dict, edge_idx)
            gnn_preds = gnn_out.cpu().numpy()

            y_true_gnn_np = y_true_gnn.cpu().numpy()

            gnn_rmse = math.sqrt(mean_squared_error(y_true_gnn_np, gnn_preds))
            gnn_mae = mean_absolute_error(y_true_gnn_np, gnn_preds)
            gnn_r2 = r2_score(y_true_gnn_np, gnn_preds)

            results.append({
                "Model": "⭐ PJFCANN Heterogeneous GNN (Đề xuất)", 
                "RMSE": round(gnn_rmse, 4), 
                "MAE": round(gnn_mae, 4), 
                "R2-Score": round(gnn_r2, 4)
            })

    # Xuất thành bảng Pandas, sắp xếp theo RMSE (càng thấp càng đứng đầu)
    df_results = pd.DataFrame(results).sort_values(by="RMSE")

    print("\n" + "="*85)
    print(f"📊 BẢNG TỔNG SẮP CHUNG CUỘC (NGÀNH {domain_name} - {model_alias})")
    print("="*85)
    print(df_results.to_markdown(index=False))

    return df_results


DOMAINS = ["IT"]
MODELS = ["MPNet", "JobBERT", "BGE", "E5", "MultiQA"]

# Từ điển để lưu lại toàn bộ bảng kết quả nếu cậu muốn xuất ra Excel sau này
all_arena_results = {}

print("🔥 BẮT ĐẦU ĐẤU TRƯỜNG AI TỔNG LỰC CHO TẤT CẢ CÁC LÕI EMBEDDING 🔥\n")

for domain in DOMAINS:
    for model_alias in MODELS:
        print(f"⚔️ ĐANG GỌI ĐẤU TRƯỜNG: NGÀNH {domain} - LÕI {model_alias}")

        # Gọi hàm run_arena và hứng kết quả (DataFrame)
        df_result = run_arena(domain, model_alias)

        # Lưu vào từ điển tổng
        all_arena_results[f"{domain}_{model_alias}"] = df_result
        print("\n")

In [ ]:
import pandas as pd
import numpy as np

# 1. ĐỊNH TUYẾN ĐƯỜNG DẪN (DÙNG MỎ VÀNG SYNTHETIC DATA)
# Đây là file tổng 1,760 dòng mà cậu vừa gộp nãy giờ
SYNTHETIC_DATA_PATH = '/kaggle/input/datasets/kevinhieutmtpv/score-v1/CV_Dataset_FAB.csv' 
JD_MASTER_PATH = '/kaggle/input/datasets/kevinhieutmtpv/score-v1/JD_Dataset_FAB.csv'

# File xuất ra cuối cùng để tống vào lò train Classification
OUTPUT_CLASSIFICATION_DATASET = '/kaggle/working/final_classification_dataset.csv'

print("🔥 Khởi động quy trình dọn cỗ cho PJFCANN Classification...")

# 2. NẠP DỮ LIỆU
df_synth = pd.read_csv(SYNTHETIC_DATA_PATH)
# Chỉ lấy các cột cần thiết từ JD
df_jd = pd.read_csv(JD_MASTER_PATH)[['Job_ID', 'TITLE', 'DESCRIPTION', 'EXPERIENCE', 'SKILLS']].fillna("")

# 3. ĐỒNG BỘ ID VÀ MERGE LẤY TEXT TỪ JD
# Bọc thép ID để đảm bảo map trúng 100%
df_synth['Job_ID'] = df_synth['Job_ID'].astype(str).str.upper().str.strip()
df_jd['Job_ID'] = df_jd['Job_ID'].astype(str).str.upper().str.strip()
df_jd = df_jd.drop_duplicates(subset=['Job_ID'])

df_merged = pd.merge(df_synth, df_jd, on='Job_ID', how='left')
df_merged = df_merged.fillna("").astype(str)

# 4. GỘP CHỮ CHO MẠNG BiLSTM "ĂN" (Feature Engineering)
print("-> Đang nhồi nhét tinh hoa ngữ nghĩa vào JD_Text và CV_Text...")
df_merged['JD_Text'] = df_merged['TITLE'] + " " + df_merged['DESCRIPTION'] + " " + df_merged['EXPERIENCE'] + " " + df_merged['SKILLS']
df_merged['CV_Text'] = df_merged['TITLE_CV'] + " " + df_merged['DESCRIPTION_CV'] + " " + df_merged['EXPERIENCE_CV']

# Ép kiểu nhãn cứng (Label_Final) về số nguyên chuẩn xác
df_merged['Label_Final'] = pd.to_numeric(df_merged['Label_Final'], errors='coerce').fillna(0).astype(int)

# 5. CHỐT HẠ KHUÔN MẪU BẮT BUỘC CHO CLASSIFICATION
final_cols = ['Job_ID', 'Resume_ID', 'JD_Text', 'CV_Text', 'Label_Final']
df_classification = df_merged[final_cols]

# [KỸ THUẬT BẮT BUỘC]: Xáo trộn (Shuffle) toàn bộ dataset
# Nếu không shuffle, mạng nơ-ron học nhãn 1 xong mới học nhãn 0 sẽ bị "ngáo" (Catastrophic Forgetting)
df_classification = df_classification.sample(frac=1, random_state=42).reset_index(drop=True)

df_classification.to_csv(OUTPUT_CLASSIFICATION_DATASET, index=False)

print("="*70)
print(f"✅ XONG! Dataset Classification đã sẵn sàng với {len(df_classification)} mẫu chuẩn mực.")
print(f"💾 Lưu tại: {OUTPUT_CLASSIFICATION_DATASET}")
print("\n📊 BẢNG PHÂN PHỐI NHÃN (ĐẬU / TRƯỢT):")
print(df_classification['Label_Final'].value_counts())
print("="*70)

# Hiển thị thử để nghiệm thu
display(df_classification.head(3))

MODULE MẠNG ĐỒ THỊ (GNN)

In [12]:
class GNN(nn.Module):
    def __init__(self, hidden_size, step=1):
        super(GNN, self).__init__()
        self.step = step
        self.hidden_size = hidden_size
        # Các trọng số cho cổng Update (z) và Reset (r) theo Công thức (8) của bài báo
        self.W_z = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.U_z = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        
        self.W_r = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.U_r = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        
        self.W_h = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.U_h = nn.Linear(self.hidden_size, self.hidden_size, bias=False)

    def forward(self, A, hidden):
        # A: Ma trận kề (Adjacency Matrix) thể hiện mối liên kết giữa các CVs và JDs
        # hidden: Các vector Node ban đầu (g_0)
        
        for i in range(self.step):
            # Bước 1: Gộp thông tin từ các node lân cận (Neighborhood Aggregation)
            # a_t = A * g_{t-1}
            a = torch.matmul(A, hidden) 
            
            # Bước 2: Tính toán Cổng Update (z) và Cổng Reset (r)
            z = torch.sigmoid(self.W_z(a) + self.U_z(hidden))
            r = torch.sigmoid(self.W_r(a) + self.U_r(hidden))
            
            # Bước 3: Tính toán trạng thái ẩn dự kiến (tilde_g)
            q = torch.tanh(self.W_h(a) + self.U_h(r * hidden))
            
            # Bước 4: Cập nhật vector cuối cùng cho node (g_t)
            hidden = (1 - z) * hidden + z * q
            
        return hidden

class SessionGraph(nn.Module):
    def __init__(self, hidden_size, n_node):
        super(SessionGraph, self).__init__()
        self.hidden_size = hidden_size
        self.n_node = n_node
        # Khởi tạo ma trận Embedding cho các Node (JD hoặc CV)
        self.embedding = nn.Embedding(self.n_node, self.hidden_size)
        self.gnn = GNN(self.hidden_size, step=1)
        
    def forward(self, A, node_indices):
        # 1. Lấy toàn bộ vector gốc của TẤT CẢ các node trong đồ thị
        all_node_emb = self.embedding.weight 
        
        # 2. Đưa vào GNN cùng ma trận kề A để các CV và JD "trao đổi thông tin" với nhau
        updated_node_emb = self.gnn(A, all_node_emb)
        
        # 3. Sau khi đồ thị đã được cập nhật, mới trích xuất vector của JD hoặc CV đang xét
        return updated_node_emb[node_indices]

MODULE NGỮ NGHĨA (SEMANTIC)

In [13]:
class SemanticEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size):
        super(SemanticEncoder, self).__init__()
        self.word_embedding = nn.Embedding(vocab_size, embed_dim)
        # Sử dụng BiLSTM theo như kết quả tốt nhất của bài báo (PJFCANN BiLSTM+Attention)
        self.bilstm = nn.LSTM(embed_dim, hidden_size // 2, num_layers=1, bidirectional=True, batch_first=True)
        self.attention = nn.Linear(hidden_size, 1)
        
    def forward(self, text_seq):
        # text_seq: [batch_size, seq_length]
        emb = self.word_embedding(text_seq) # [batch_size, seq_length, embed_dim]
        out, _ = self.bilstm(emb)           # [batch_size, seq_length, hidden_size]
        
        # Cơ chế Soft-Attention để tìm các từ khóa quan trọng
        attn_weights = F.softmax(self.attention(out), dim=1) 
        context_vector = torch.sum(attn_weights * out, dim=1) # [batch_size, hidden_size]
        return context_vector

MÔ HÌNH PJFCANN

In [ ]:
class PJFCANN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, n_J_node, n_R_node):
        super(PJFCANN, self).__init__()
        self.hidden_size = hidden_size
        
        # 3.1 Nhánh Semantic (Local Representation)
        self.semantic_encoder = SemanticEncoder(vocab_size, embed_dim, hidden_size)
        
        # 3.2 Nhánh GNN (Global Experience Representation)
        self.gnn_J = SessionGraph(hidden_size, n_J_node)
        self.gnn_R = SessionGraph(hidden_size, n_R_node)
        
        # 3.3 Cơ chế Co-Attention ghép nối kinh nghiệm
        self.W_J = nn.Linear(2 * hidden_size, hidden_size)
        self.W_R = nn.Linear(2 * hidden_size, hidden_size)
        
        # 3.4 Lớp phân loại cuối cùng (Classifier)
        # Vector đầu vào = Local(JD) + Global(JD) + Local(CV) + Global(CV) + Sự khác biệt
        self.fc1 = nn.Linear(hidden_size * 6, 512) 
        self.fc2 = nn.Linear(512, 1)
        
    def forward(self, jd_text, cv_text, jd_idx, cv_idx, A_J, A_R):
        # Bước 1: Trích xuất Ngữ nghĩa (Local)
        H_local_J = self.semantic_encoder(jd_text)
        H_local_R = self.semantic_encoder(cv_text)
        
        # Bước 2: Trích xuất Kinh nghiệm Đồ thị (Global)
        G_J = self.gnn_J(A_J, jd_idx)
        G_R = self.gnn_R(A_R, cv_idx)
        
        # Kết hợp Local và Global thông qua hàm Tanh 
        H_global_J = torch.tanh(self.W_J(torch.cat([H_local_J, G_J], dim=1)))
        H_global_R = torch.tanh(self.W_R(torch.cat([H_local_R, G_R], dim=1)))
        
        # Bước 3: Person-Job Fit Label Prediction
        H_J = torch.cat([H_local_J, H_global_J], dim=1)
        H_R = torch.cat([H_local_R, H_global_R], dim=1)
        
        # Tính toán độ chênh lệch (H_J - H_R)
        diff = torch.abs(H_J - H_R)
        
        D = torch.tanh(self.fc1(torch.cat([H_J, H_R, diff], dim=1)))
        out = torch.sigmoid(self.fc2(D)) # Output từ 0 đến 1
        
        return out

print("✅ Đã khởi tạo thành công cấu trúc kiến trúc PJFCANN")

In [15]:
BALANCED_DATA_PATH = '/kaggle/working/final_classification_dataset.csv'
OUTPUT_DIR = '/kaggle/working/PJFCANN_Data/'
if not os.path.exists(OUTPUT_DIR):
    print("Vào if")
    os.makedirs(OUTPUT_DIR)
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, 'pjfcann_best_model.pth')
MAX_JD_LEN = 100  
MAX_CV_LEN = 600    
BATCH_SIZE = 16        
EMBED_DIM = 200        
HIDDEN_SIZE = 200      
EPOCHS = 10
LEARNING_RATE = 0.001

XÂY DỰNG TỪ VỰNG

In [ ]:
import pandas as pd
from collections import Counter
import json

# 1. NẠP DATA ĐÃ LÀM SẠCH (KHÔNG CẦN MERGE NỮA)
df = pd.read_csv(BALANCED_DATA_PATH)

# Đảm bảo Text là chuỗi để không bị lỗi lúc tách từ
df['JD_Text'] = df['JD_Text'].astype(str)
df['CV_Text'] = df['CV_Text'].astype(str)

# Cắt khoảng trắng ID cho chắc chắn
df['Job_ID'] = df['Job_ID'].astype(str).str.strip()
df['Resume_ID'] = df['Resume_ID'].astype(str).str.strip()

display(df.head(3))

# 2. XÂY DỰNG TỪ ĐIỂN CHỮ (VOCABULARY CHO TENSOR)
print("\n-> Đang xây dựng từ điển (Vocabulary)...")
vocab = Counter()
# Lưu ý: Cột text mới của ta tên là JD_Text và CV_Text
for text in df['JD_Text'].tolist() + df['CV_Text'].tolist():
    vocab.update(str(text).lower().split())

word2idx = {'<PAD>': 0, '<UNK>': 1}
for word, _ in vocab.items():
    if word not in word2idx:
        word2idx[word] = len(word2idx)
        
VOCAB_SIZE = len(word2idx)
print(f"-> Tổng số từ vựng (Vocab Size): {VOCAB_SIZE}")

vocab_out_path = '/kaggle/working/word2idx.json'
with open(vocab_out_path, 'w', encoding='utf-8') as f:
    json.dump(word2idx, f, ensure_ascii=False)
print(f"-> 💾 Đã lưu từ điển ra file để deploy: {vocab_out_path}")

print("\n-> Đang tạo bộ từ điển Indexing (ID sang Number) động...")
unique_jds = df['Job_ID'].unique().tolist()
unique_cvs = df['Resume_ID'].unique().tolist()

jid2num = {jid: int(idx) for idx, jid in enumerate(unique_jds)}
uid2num = {uid: int(idx) for idx, uid in enumerate(unique_cvs)}

print(f"-> ✅ Đã mã hóa thành công {len(unique_jds)} JDs và {len(unique_cvs)} CVs.")

In [ ]:
# THỐNG KÊ SỐ LƯỢNG TỪ TRONG JD VÀ CV
print("\n--- ĐANG PHÂN TÍCH CHIỀU DÀI VĂN BẢN ---")

# Tính số từ (cắt theo khoảng trắng) cho từng hàng
df['JD_word_count'] = df['JD_Full_Text'].apply(lambda x: len(str(x).split()))
df['CV_word_count'] = df['CV_Full_Text'].apply(lambda x: len(str(x).split()))

df.info()

print("\n1. Thống kê chiều dài Job Description (JD):")
# In ra các mốc % quan trọng
print(df['JD_word_count'].describe(percentiles=[0.5, 0.75, 0.85, 0.90, 0.95]).astype(int))

print("\n2. Thống kê chiều dài Resume (CV):")
print(df['CV_word_count'].describe(percentiles=[0.5, 0.75, 0.85, 0.90, 0.95]).astype(int))
print("=========================================================\n")

XÂY DỰNG DATALOADER

In [20]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torch

class PersonJobFitDataset(Dataset):
    def __init__(self, dataframe, word2idx, jid2num, uid2num, max_jd_len, max_cv_len):
        self.data = dataframe
        self.word2idx = word2idx
        self.jid2num = jid2num
        self.uid2num = uid2num
        self.max_jd_len = max_jd_len
        self.max_cv_len = max_cv_len

    def tokenize_and_pad(self, text, target_len):
        words = str(text).lower().split()
        seq = [self.word2idx.get(w, self.word2idx['<UNK>']) for w in words]
        if len(seq) < target_len:
            seq += [self.word2idx['<PAD>']] * (target_len - len(seq))
        else:
            seq = seq[:target_len]
        return torch.tensor(seq, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # Text to Tensor
        jd_text = self.tokenize_and_pad(row['JD_Text'], self.max_jd_len)
        cv_text = self.tokenize_and_pad(row['CV_Text'], self.max_cv_len)
        
        # ID to Node Index (cho GNN)
        jd_idx = torch.tensor(self.jid2num[str(row['Job_ID']).strip()], dtype=torch.long)
        cv_idx = torch.tensor(self.uid2num[str(row['Resume_ID']).strip()], dtype=torch.long)
        
        # Label (0 hoặc 1) Label_Final
        # label = torch.tensor([float(row['Label_Y'])], dtype=torch.float32)
        label = torch.tensor([float(row['Label_Final'])], dtype=torch.float32)
        
        return jd_text, cv_text, jd_idx, cv_idx, label

# Chia tập Train/Test (80% Train, 20% Test)
# train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Label_Y'])
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Label_Final'])

train_dataset = PersonJobFitDataset(train_df, word2idx, jid2num, uid2num, MAX_JD_LEN, MAX_CV_LEN)
test_dataset = PersonJobFitDataset(test_df, word2idx, jid2num, uid2num, MAX_JD_LEN, MAX_CV_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

THIẾT LẬP MÔ HÌNH, OPTIMIZER & LOSS FUNCTION

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang chạy mô hình trên thiết bị: {device}")

# Khởi tạo mô hình PJFCANN 
model = PJFCANN(vocab_size=VOCAB_SIZE, 
                embed_dim=EMBED_DIM, 
                hidden_size=HIDDEN_SIZE, 
                n_J_node=len(jid2num), 
                n_R_node=len(uid2num)).to(device)

# Sử dụng Binary Cross Entropy Loss và Adam Optimizer
criterion = nn.BCELoss() 
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)

TẠO MA TRẬN KỀ CHO GNN (A_J và A_R)

In [ ]:
print("\n--- ĐANG XÂY DỰNG MA TRẬN ĐỒ THỊ LỊCH SỬ (A_J và A_R) ---")
n_J = len(jid2num)
n_R = len(uid2num)

# Khởi tạo ma trận đường chéo (Self-loop: Tự kết nối với chính mình)
A_J_np = np.eye(n_J, dtype=np.float32) 
A_R_np = np.eye(n_R, dtype=np.float32)
# Label_Final
# df_success = train_df[train_df['Label_Y'] == 1]
df_success = train_df[train_df['Label_Final'] == 1]

# 1. Đồ thị J-J: Các Job Postings có chung 1 CV trúng tuyển 
for cv_id, group in df_success.groupby('Resume_ID'):
    j_list = [jid2num[str(x).strip()] for x in group['Job_ID'].tolist() if str(x).strip() in jid2num]
    for i in j_list:
        for j in j_list:
            A_J_np[i, j] = 1.0

# 2. Đồ thị R-R: Các CV trúng tuyển vào chung 1 Job Posting
for jd_id, group in df_success.groupby('Job_ID'):
    r_list = [uid2num[str(x).strip()] for x in group['Resume_ID'].tolist() if str(x).strip() in uid2num]
    for i in r_list:
        for j in r_list:
            A_R_np[i, j] = 1.0

# Cộng dồn từng hàng (axis=1) để biết mỗi Node có bao nhiêu hàng xóm
degree_J = A_J_np.sum(axis=1, keepdims=True)
degree_R = A_R_np.sum(axis=1, keepdims=True)

# Tránh lỗi chia cho 0 bằng cách thay thế các số 0 thành 1
degree_J[degree_J == 0] = 1.0
degree_R[degree_R == 0] = 1.0

# Chia mỗi hàng cho tổng của hàng đó
A_J_np = A_J_np / degree_J
A_R_np = A_R_np / degree_R

# Chuyển đổi thành Tensor và đưa vào thiết bị (GPU/CPU)
A_J = torch.tensor(A_J_np).to(device)
A_R = torch.tensor(A_R_np).to(device)
print("-> Đã tạo xong Ma trận kề A_J và A_R. Sẵn sàng huấn luyện!")

TRAINING LOOP và EVALUATION

In [ ]:
print("\n--- BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH PJFCANN ---")
best_f1 = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    
    # [TRAIN PHASE]
    for batch_idx, (jd_text, cv_text, jd_idx, cv_idx, labels) in enumerate(train_loader):
        jd_text, cv_text = jd_text.to(device), cv_text.to(device)
        jd_idx, cv_idx, labels = jd_idx.to(device), cv_idx.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(jd_text, cv_text, jd_idx, cv_idx, A_J, A_R)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    avg_train_loss = total_loss / len(train_loader)
    
    # [EVALUATION PHASE - Dựa theo score.py của bài báo]
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for jd_text, cv_text, jd_idx, cv_idx, labels in test_loader:
            jd_text, cv_text = jd_text.to(device), cv_text.to(device)
            jd_idx, cv_idx = jd_idx.to(device), cv_idx.to(device)
            
            outputs = model(jd_text, cv_text, jd_idx, cv_idx, A_J, A_R)
            
            # Làm tròn output (sigmoid) để lấy nhãn 0 hoặc 1
            preds = (outputs > 0.5).float().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            
    # Tính toán các chỉ số theo chuẩn bài báo
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    
    print(f"Epoch [{epoch}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Test Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f}")
    
    # Lưu trọng số mô hình tốt nhất (Best Checkpoint)
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"  -> [Đã lưu Model tốt nhất với F1 Score: {best_f1:.4f}]")

print(f"\n✅ ĐÃ HOÀN TẤT HUẤN LUYỆN! Trọng số mô hình tốt nhất lưu tại: {MODEL_SAVE_PATH}")